In [17]:
# the use of this code is intended for organisacional purposes
# don't use it in production

import pandas as pd
from random import choice, randint

print("done.")

done.


In [ ]:
# client data, correction of pw field

PROHIBITED = " \\ ' { } ( ) [ ] / ~ ` | , * > < # . - + "

PERMITTED = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789@_"

df_people = pd.read_csv("people.csv")

def clean_password(pw):
    pw = str(pw)
    new_pw = ""
    for c in pw:
        if c in PROHIBITED or c == '"' :
            c = choice(PERMITTED)
        new_pw += c
    return new_pw

def assign_class(row):
    num = randint(1, 100)
    if num < 70:
        return "client"
    elif num < 90:
        return "delivery"
    else:
        return "employee"


df_people["password"] = df_people["password"].apply(clean_password)
df_people["class"] = df_people.apply(assign_class, axis=1)
df_people.to_csv("people.csv", index=False)

df_people.head()

,id,name,password,street_name,street_number,postal_code,city,telefon_number,class
0,1001,Trefor,xU1ucaWtOeLR,Orin,10,453550,Tukan,539-502-0149,delivery
1,1002,Persis,pR6UV0B54DDyan$,Ohio,35770,63610-000,Mombaça,971-728-5245,client
2,1003,Oberon,sE9%8FTfOiWYG,Columbus,83377,32595,Pensacola,407-225-7304,client
3,1004,Kimbell,xC6GKA_2gbY&ES9,Oak,4953,987-0352,Susaki,692-181-7388,client
4,1005,Elsie,zF9rJTH_7X,Harbort,55007,5510,Lalmanirhat,994-997-1216,client


In [1]:
# create the hashes for login in a separated file

import pandas as pd
from hashlib import pbkdf2_hmac
from os import urandom
from base64 import b64encode  # b64decode when execute login
from random import randint
from datetime import datetime


# def make_password_hash(salt = urandom(16), iter = 100_000, plain_password: str = "") -> str:
#     salt = salt
#     iterations = iter
#     derived = pbkdf2_hmac(
#         "sha256",
#         plain_password.encode("utf-8"),
#         salt,
#         iterations,
#     )
#     return f"pbkdf2_sha256${iterations}${salt.hex()}${derived.hex()}"

# FIELDS FOR A PW DB
# - - - - - - - - - 
# user_id: string,
# password_hash: string,
# failed_attempts: integer,
# locked_until: unix timestamp,
# created_at: unix timestamp,
# updated_at: unix timestamp

df_people = pd.read_csv("people.csv")
df_clients = df_people[df_people["class"] == "client"]
df_clients.head()

now = datetime.now().timestamp()

pw_db_records = []
for index, row in df_clients.iterrows():

    # Generate the requested timestamps
    created_at = now - randint(60, 86400 * 30)
    updated_at = created_at + randint(1, 86400 * 7)
    
    # Hash the raw password stored in the DataFrame row
    salt = urandom(16)  # bytes
    iters = 1000  # normally a very larger number e.g. 100_000 or 200_000
    raw_pw = str(row["password"])

    hash_bytes = pbkdf2_hmac("sha256", raw_pw.encode(), salt, iters)

    salt_b64 = b64encode(salt).decode("ascii")
    hash_b64 = b64encode(hash_bytes).decode("ascii")

    # salt = urandom(16)
    # iter = 1000
    # password_hash = make_password_hash(salt, iter, raw_password)
    
    # Construct the dictionary matching your schema fields
    db_record = {
        "user_id": str(row["id"]),  # Assumes your CSV has an identifier column like "id"
        "iter": iters,
        "salt": salt_b64,
        "password_hash": hash_b64,
        "failed_attempts": 0,       # Initializing new DB tracking at 0
        "locked_until": 0,          # 0 means not locked/no timestamp restriction
        "created_at": created_at,
        "updated_at": updated_at
    }
    
    pw_db_records = pw_db_records + [db_record]

df_pw_db = pd.DataFrame(pw_db_records)
df_pw_db.to_csv("password_hashes.csv", index=False)
df_pw_db.head()




,user_id,iter,salt,password_hash,failed_attempts,locked_until,created_at,updated_at
0,1002,1000,lcaIa9uZPrf1VtJMJ7QCiA==,xAWd450MkABmmKp2slF6EdP4vZFe0gvTGKCG8jO61Dc=,0,0,1.783973e+09,1.784171e+09
1,1003,1000,eyJI02llE4psZTgBck45AA==,nY41a3sAwBsBW+JiEfCYXYxB3snvwUg2OUscs+h4fWc=,0,0,1.783763e+09,1.784368e+09
2,1004,1000,kfcncKzjz6yLQN2ZWOc9fg==,p0UhSCHudW6OFRAtlAmeRA8sgNv47Q88kyG74BRI0fQ=,0,0,1.782236e+09,1.782596e+09
3,1005,1000,5aPQIkaDP5r0MxnCP7qtEA==,d7ad0d90OqJlcT3tNhhGqBYxX/GHmctyMCoA2TR8vn0=,0,0,1.784122e+09,1.784685e+09
4,1007,1000,G/NNh9pkCaltsIhXylnd3w==,+Nc/hJLh34Wf8B3tDbI8beN7nlsBU9r0pFkM55JoRic=,0,0,1.783935e+09,1.784493e+09
